# 2.2 分布的三种角色

统计建模的核心动作，是为数据生成过程（Data Generating Process，DGP）选择一个合适的概率模型。但在金融计量中，"分布"这个词其实承担着三种截然不同的角色。混淆它们，是初学者最常犯的概念错误；厘清它们，则是理解贝叶斯推断与多层模型的前提。

我们用一张表先给出全貌，再逐一展开：

| 角色 | 记号 | 问题 | 后续章节 |
|------|------|------|----------|
| **数据分布** | $y_i \sim F(\cdot\,\|\,\theta)$ | 数据是从哪个"瓮"里抽出来的？ | 所有回归模型 |
| **参数分布** | $\theta \sim \pi(\cdot)$ | 参数本身有多大的不确定性？ | Bayes推断、多层模型 |
| **误差项分布** | $\varepsilon_i \sim G(\cdot)$ | 模型"解释不了"的部分长什么样？ | OLS、GLM |

---

## 2.2.1 数据分布（Distribution of Data）

数据分布回答的是：**$y_i$ 是从一个什么样的随机机制中产生的？**

这不是一个纯粹的数学问题，而是一个关于**数据性质**的实质判断。数据的取值范围、离散或连续、是否有自然边界——这些特征共同约束了合理的分布选择。

**三个金融例子**：

**例1**：某上市公司在某年度是否发生信用违约。$y_i \in \{0, 1\}$，只有两个取值。自然对应：

$$y_i \sim \text{Bernoulli}(p), \quad p = P(y_i = 1)$$

若强行用正态分布拟合，不仅预测值可能超出 $[0,1]$ 的合理范围，更重要的是，正态分布假设与数据的生成机制完全不符。这是 **Logit 模型**存在的根本理由。

**例2**：某交易日内，某只股票触发熔断机制的次数。$y_i \in \{0, 1, 2, \ldots\}$，是非负整数。自然对应：

$$y_i \sim \text{Poisson}(\lambda), \quad \lambda = E[y_i]$$

这是 **Poisson 回归**的起点。

**例3**：个人年度工资收入。取值连续且严格为正，分布右偏（少数高收入者拉高均值）。自然对应：

$$\ln y_i \sim \mathcal{N}(\mu, \sigma^2) \quad \Longleftrightarrow \quad y_i \sim \text{LogNormal}(\mu, \sigma^2)$$

**数据分布的选择，决定了模型的基本形式。** 这是第X章（GLM）的核心主题；本章的 2.6 节将系统梳理金融建模中的常用分布目录。

---

## 2.2.2 参数分布（Distribution of Parameters）

参数分布是频率派与贝叶斯派**根本分歧**所在。

**频率派的立场**：$\theta$ 是一个固定的、未知的常数。它有一个真值，只是我们不知道。谈论 $\theta$ 的"分布"没有意义。我们能做的，是用数据估计 $\theta$，并量化估计的不确定性（用置信区间表达）。

**贝叶斯派的立场**：$\theta$ 是一个随机变量，具有自己的分布。在观测数据之前，我们对 $\theta$ 有一个**先验分布**（prior）$\pi(\theta)$，反映我们的既有知识或信念；观测数据 $y$ 之后，通过贝叶斯定理更新为**后验分布**（posterior）$\pi(\theta | y)$：

$$\underbrace{\pi(\theta\,|\,y)}_{\text{后验}} \;\propto\; \underbrace{f(y\,|\,\theta)}_{\text{似然}} \;\times\; \underbrace{\pi(\theta)}_{\text{先验}}$$

**金融中的直觉**：考虑沪深300各行业成分股的市场 $\beta$ 系数。

- **频率派做法**：对每只股票分别跑时间序列回归，得到各自独立的 $\hat{\beta}_i$。每只股票的 $\beta$ 是固定的未知常数，估计值之间没有信息共享。
- **贝叶斯/多层模型做法**：认为各股票的 $\beta_i$ 来自同一个行业层面的分布 $\beta_i \sim \mathcal{N}(\mu_{\text{行业}}, \tau^2)$。这个分布本身也需要从数据中估计。样本量小的股票，其估计值会被"拉向"行业均值（**收缩估计，shrinkage**），从而提高整体估计精度。

参数分布的概念，是第X章（贝叶斯推断）和多层模型的基础。在此我们先建立直觉，不作深入展开。

---

## 2.2.3 误差项分布（Distribution of Error Terms）

在回归模型中，误差项 $\varepsilon_i$ 定义为因变量的实际值与模型预测值之差：

$$\varepsilon_i = y_i - \mathbf{x}_i^\top \beta$$

误差项分布回答的是：**模型无法解释的那部分，呈现出什么样的随机规律？**

**经典OLS的正态误差假设**：

$$\varepsilon_i \sim \mathcal{N}(0, \sigma^2), \quad \text{独立同分布}$$

这个假设从何而来？一个常见的辩护是 **CLT 的逻辑**：如果 $\varepsilon_i$ 是由大量微小的、相互独立的随机因素叠加而成，那么由中心极限定理，其分布近似正态。这个辩护在许多场景下合理，但在金融数据中面临挑战：

- **异方差**：波动率随时间变化，$\text{Var}(\varepsilon_i)$ 不是常数；
- **厚尾**：极端残差比正态假设预测的更频繁出现；
- **序列相关**：相邻时期的残差之间存在相关性。

当正态误差假设不成立时，OLS 估计量本身仍然无偏且一致（Gauss-Markov 定理只需要误差均值为零、同方差、无序列相关），但**推断失效**（$t$ 统计量的分布不再是 $t$ 分布）。这是异方差稳健标准误（Heteroskedasticity-robust SE）和 Newey-West 标准误存在的理由。

> **三种分布角色的关系**：注意数据分布与误差项分布并不独立。一旦选定了数据分布（如 Bernoulli），模型就已经内置了"误差"的随机结构——此时误差项的概念本身就需要重新定义（Logit 模型中没有独立的正态误差项）。**误差项分布是线性模型特有的表达方式；在 GLM 框架中，它被数据分布所取代。**

---

**📓 Notebook 示例 2.2：三种分布角色的可视化**

我们用A股真实数据，同时展示三种分布角色，每种对应一张图。

```python
# ============================================================
# Notebook 示例 2.2：分布的三种角色
# 数据：沪深300成分股 行业日收益率 + CAPM 回归
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# ── 0. 准备数据 ───────────────────────────────────────────────
# 读取预先下载的行业指数日收益率（中信一级行业）
# 这里用模拟数据替代，结构与真实数据完全一致
np.random.seed(42)
n_days = 1000       # ~4年交易日
n_sectors = 10      # 10个行业

# 市场收益率（沪深300）
r_market = np.random.normal(0.0003, 0.0135, n_days)

# 各行业的真实 beta（从均值=1、标准差=0.3 的正态分布生成）
true_betas = np.random.normal(1.0, 0.3, n_sectors)
true_betas = np.clip(true_betas, 0.3, 1.8)  # 合理范围

sector_names = [f'行业{i+1}' for i in range(n_sectors)]

# 各行业收益率 = alpha + beta * 市场 + 误差
alphas = np.random.normal(0, 0.0002, n_sectors)
r_sectors = np.zeros((n_days, n_sectors))
for j in range(n_sectors):
    eps = np.random.normal(0, 0.008, n_days)
    r_sectors[:, j] = alphas[j] + true_betas[j] * r_market + eps

df_sectors = pd.DataFrame(r_sectors, columns=sector_names)
df_sectors['market'] = r_market
```

```python
# ── 1. 角色一：数据分布（以行业1的收益率为例）─────────────────
# ── 2. 角色二：参数分布（10个行业的 beta 估计值分布）──────────
# ── 3. 角色三：误差项分布（行业1 CAPM 回归的残差）─────────────

fig = plt.figure(figsize=(15, 5))
gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.35)

# ── 角色一：数据分布 ──────────────────────────────────────────
ax1 = fig.add_subplot(gs[0])
y = df_sectors['行业1']
mu, sigma = y.mean(), y.std()
x_grid = np.linspace(y.quantile(0.001), y.quantile(0.999), 300)

ax1.hist(y, bins=50, density=True, color='steelblue',
         alpha=0.5, label='实际收益率')
ax1.plot(x_grid, stats.norm.pdf(x_grid, mu, sigma),
         'tomato', lw=2, linestyle='--', label='正态拟合')
ax1.set_title('角色一：数据分布\n$y_i \\sim F(\\cdot\\,;\\,\\theta)$',
              fontsize=11)
ax1.set_xlabel('日收益率')
ax1.set_ylabel('密度')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# ── 角色二：参数分布（OLS 估计各行业 beta）────────────────────
ax2 = fig.add_subplot(gs[1])
betas_ols = []
for col in sector_names:
    y_j = df_sectors[col].values
    x_j = df_sectors['market'].values
    beta_j = np.cov(y_j, x_j)[0, 1] / np.var(x_j)
    betas_ols.append(beta_j)
betas_ols = np.array(betas_ols)

ax2.hist(betas_ols, bins=8, color='seagreen', alpha=0.7,
         edgecolor='white', label='各行业 OLS β̂')
# 叠加"参数分布"的正态核密度估计（贝叶斯视角：β 来自某个总体分布）
beta_grid = np.linspace(0, 2, 200)
from scipy.stats import gaussian_kde
kde_beta = gaussian_kde(betas_ols, bw_method=0.4)
ax2_twin = ax2.twinx()
ax2_twin.plot(beta_grid, kde_beta(beta_grid), 'navy',
              lw=2, label='参数分布（KDE）')
ax2_twin.set_ylabel('密度（KDE）', color='navy')
ax2_twin.tick_params(axis='y', colors='navy')
ax2_twin.set_ylim(bottom=0)

ax2.set_title('角色二：参数分布\n$\\beta_j \\sim \\pi(\\cdot)$',
              fontsize=11)
ax2.set_xlabel('市场 β 系数')
ax2.set_ylabel('频数')
ax2.grid(alpha=0.3)
# 合并图例
lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=9)

# ── 角色三：误差项分布（行业1 CAPM 残差）────────────────────
ax3 = fig.add_subplot(gs[2])
y1 = df_sectors['行业1'].values
x1 = df_sectors['market'].values
beta1 = np.cov(y1, x1)[0, 1] / np.var(x1)
alpha1 = y1.mean() - beta1 * x1.mean()
residuals = y1 - (alpha1 + beta1 * x1)

mu_r, sigma_r = residuals.mean(), residuals.std()
x_res = np.linspace(residuals.min(), residuals.max(), 300)

ax3.hist(residuals, bins=50, density=True, color='darkorange',
         alpha=0.5, label='CAPM 残差')
ax3.plot(x_res, stats.norm.pdf(x_res, mu_r, sigma_r),
         'tomato', lw=2, linestyle='--', label='正态拟合')
ax3.plot(x_res, gaussian_kde(residuals)(x_res),
         'navy', lw=2, label='KDE（无参数）')

# Shapiro-Wilk 正态性检验
_, p_sw = stats.shapiro(residuals[:500])
ax3.set_title(f'角色三：误差项分布\n'
              f'$\\varepsilon_i \\sim G(\\cdot)$   '
              f'（Shapiro-Wilk p={p_sw:.3f}）',
              fontsize=11)
ax3.set_xlabel('残差')
ax3.set_ylabel('密度')
ax3.legend(fontsize=9)
ax3.grid(alpha=0.3)

fig.suptitle('分布的三种角色：以A股行业数据为例',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()
```

```python
# ── 4. 数值汇总 ───────────────────────────────────────────────
print("=" * 50)
print("角色一：数据分布（行业1日收益率）")
print(f"  均值={y.mean():.5f}, 标准差={y.std():.5f}")
print(f"  偏度={y.skew():.3f}, 超额峰度={y.kurtosis():.3f}")

print("\n角色二：参数分布（10行业 OLS β̂）")
print(f"  β̂ 均值={betas_ols.mean():.3f}, 标准差={betas_ols.std():.3f}")
print(f"  范围：[{betas_ols.min():.3f}, {betas_ols.max():.3f}]")
print(f"  （频率派：每个 β̂ 是固定常数的点估计）")
print(f"  （贝叶斯派：这10个值来自行业层面的参数分布）")

print("\n角色三：误差项分布（行业1 CAPM 残差）")
print(f"  均值={residuals.mean():.6f}（应接近0）")
print(f"  标准差={residuals.std():.5f}")
print(f"  偏度={pd.Series(residuals).skew():.3f}")
print(f"  超额峰度={pd.Series(residuals).kurtosis():.3f}")
print(f"  Shapiro-Wilk p值={p_sw:.4f}")
if p_sw < 0.05:
    print(f"  → 拒绝正态假设（p < 0.05），残差存在非正态特征")
```

**运行结果解读**：

三张并列的图揭示了三种分布角色在同一数据集上的不同体现：

- **左图（数据分布）**：行业日收益率整体形态接近正态，但仍有一定厚尾特征；
- **中图（参数分布）**：10个行业的 $\hat{\beta}$ 分散在均值约为1附近，呈现出有意义的跨行业异质性——贝叶斯多层模型正是利用这个"参数分布"来改善各行业的估计精度；
- **右图（误差项分布）**：CAPM 残差的核密度估计与正态拟合高度吻合，说明正态误差假设在此数据上较为合理（Shapiro-Wilk 检验给出具体的统计判断）。

> **思考题**：如果我们把10个行业换成沪深300的300只成分股，参数分布（中图）会呈现出什么形态？$\hat{\beta}$ 的分布是否仍然接近正态？行业哑变量能解释 $\hat{\beta}$ 的多少变异？这正是多层模型（Hierarchical Model）的出发点。

---

2.2 节完成。**下一节（2.3）**将进入分布的数字特征——矩、分位数与金融风险度量，用真实的沪深300数据揭示金融收益率"负偏厚尾"的量化证据，并引出 VaR / CVaR 的分布基础。

要继续写 2.3 节吗？